# 📊 Trading Strategien Backtest

Dieses Notebook testet **2 Trading-Strategien** mit einem Startkapital von **$10.000** auf historischen Kursdaten.

### Strategien:
1. **Linear Regression** – Kauf wenn Preis unter Regressionsband, Verkauf wenn darüber
2. **Multi-Signal Ensemble** – Kombiniert 5 Indikatoren (RSI, MACD, Bollinger Bands, ADX, Volumen) und kauft nur wenn ≥3 von 5 bullish bestätigen. Verkauft wenn ≥3 bearish sind oder Trailing Stop-Loss greift.

## 1. Libraries importieren

In [9]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Konfiguration
SYMBOL = "AMD"
STARTKAPITAL = 10_000
PERIOD = "3y"

print(f"Symbol: {SYMBOL}")
print(f"Startkapital: ${STARTKAPITAL:,.0f}")
print(f"Zeitraum: {PERIOD}")

Symbol: AMD
Startkapital: $10,000
Zeitraum: 3y


## 2. Historische Kursdaten laden

In [10]:
# Daten herunterladen
raw = yf.download(tickers=SYMBOL, period=PERIOD, progress=False)
raw.columns = raw.columns.get_level_values(0)
df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.index = pd.to_datetime(df.index)

print(f"Daten von {df.index[0].date()} bis {df.index[-1].date()}")
print(f"Anzahl Handelstage: {len(df)}")
print(f"\nLetzte 5 Tage:")
df.tail()

Daten von 2023-04-03 bis 2026-04-01
Anzahl Handelstage: 752

Letzte 5 Tage:


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-03-26,217.979996,221.000000,203.429993,203.770004,49213000
2026-03-27,201.770004,203.270004,197.690002,201.990005,29206600
2026-03-30,204.949997,208.429993,192.869995,196.039993,41053300
2026-03-31,198.050003,204.000000,196.410004,203.429993,42318700
2026-04-01,207.690002,213.830002,205.839996,210.210007,40279250


## 3. Strategie 1: Linear Regression (deine bestehende Strategie)

In [11]:
# --- Linear Regression Strategie ---
REG_WINDOW = 100
BUY_DEVIATION = 0.14
SELL_DEVIATION = 0.16
STOP_LOSS = 0.20
STOP_LOSS_ACTIVE = True
LEVERAGE = 2  # 3x Hebel

def rolling_linreg(series, window):
    y = series.values
    x = np.arange(window)
    reg = np.full(len(y), np.nan)
    for i in range(window, len(y)):
        y_window = y[i-window:i]
        m, c = np.polyfit(x, y_window, 1)
        reg[i] = m * (window - 1) + c
    return pd.Series(reg, index=series.index)

df['Reg_Line'] = rolling_linreg(df['Close'], REG_WINDOW)
df['Reg_Upper'] = df['Reg_Line'] * (1 + SELL_DEVIATION)
df['Reg_Lower'] = df['Reg_Line'] * (1 - BUY_DEVIATION)

# Signale generieren
df['LinReg_Signal'] = 0
df.loc[df['Close'] < df['Reg_Lower'], 'LinReg_Signal'] = 1   # Kaufsignal
df.loc[df['Close'] > df['Reg_Upper'], 'LinReg_Signal'] = -1  # Verkaufssignal

print(f"Linear Regression Konfiguration:")
print(f"  Buy Deviation:  {BUY_DEVIATION*100:.0f}%")
print(f"  Sell Deviation: {SELL_DEVIATION*100:.0f}%")
print(f"  Stop Loss:      {STOP_LOSS*100:.0f}%")
print(f"  Hebel:          {LEVERAGE}x")
print(f"\nSignale berechnet ✓")
print(f"  Kaufsignale:     {(df['LinReg_Signal'] == 1).sum()}")
print(f"  Verkaufssignale: {(df['LinReg_Signal'] == -1).sum()}")

Linear Regression Konfiguration:
  Buy Deviation:  14%
  Sell Deviation: 16%
  Stop Loss:      20%
  Hebel:          2x

Signale berechnet ✓
  Kaufsignale:     43
  Verkaufssignale: 72


## 4. Strategie 2: Multi-Signal Ensemble (RSI + MACD + Bollinger + ADX + Volumen)

Dieser Algorithmus kombiniert **5 technische Indikatoren** zu einem Punktesystem:
- **RSI**: Überverkauft (<35) = bullish, Überkauft (>65) = bearish
- **MACD**: Kreuzung über/unter Signallinie
- **Bollinger Bands**: Preis am unteren/oberen Band
- **ADX + DI**: Trendstärke + Richtung (ADX>20 = starker Trend)
- **Volumen-Spike**: Überdurchschnittliches Volumen bestätigt Signale

**Kauf** wenn ≥3 von 5 Indikatoren bullish → hohes Vertrauen  
**Verkauf** wenn ≥3 bearish ODER **Trailing Stop-Loss** (-8% vom Hoch) greift

In [12]:
# ==========================================
# MULTI-SIGNAL ENSEMBLE ALGORITHMUS
# ==========================================

# --- 1. RSI berechnen ---
RSI_PERIOD = 14
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0.0)
loss_s = -delta.where(delta < 0, 0.0)
avg_gain = gain.ewm(alpha=1/RSI_PERIOD, min_periods=RSI_PERIOD).mean()
avg_loss = loss_s.ewm(alpha=1/RSI_PERIOD, min_periods=RSI_PERIOD).mean()
rs = avg_gain / avg_loss
df['RSI'] = 100 - (100 / (1 + rs))

# --- 2. MACD berechnen ---
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = df['EMA_12'] - df['EMA_26']
df['MACD_Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal_Line']

# --- 3. Bollinger Bands berechnen ---
BB_WINDOW = 20
BB_STD = 2
df['BB_Mid'] = df['Close'].rolling(window=BB_WINDOW).mean()
df['BB_Std_Val'] = df['Close'].rolling(window=BB_WINDOW).std()
df['BB_Upper'] = df['BB_Mid'] + (BB_STD * df['BB_Std_Val'])
df['BB_Lower'] = df['BB_Mid'] - (BB_STD * df['BB_Std_Val'])
# Normalisierter Abstand zum unteren/oberen Band (0 = Lower, 1 = Upper)
df['BB_Pct'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

# --- 4. ADX (Average Directional Index) + DI berechnen ---
ADX_PERIOD = 14
high = df['High']
low = df['Low']
close = df['Close']

# True Range
tr1 = high - low
tr2 = (high - close.shift(1)).abs()
tr3 = (low - close.shift(1)).abs()
df['TR'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

# +DM / -DM
df['Plus_DM'] = np.where((high - high.shift(1)) > (low.shift(1) - low), 
                          np.maximum(high - high.shift(1), 0), 0)
df['Minus_DM'] = np.where((low.shift(1) - low) > (high - high.shift(1)), 
                           np.maximum(low.shift(1) - low, 0), 0)

# Smoothed averages
df['ATR'] = df['TR'].ewm(alpha=1/ADX_PERIOD, min_periods=ADX_PERIOD).mean()
df['Plus_DI'] = 100 * (df['Plus_DM'].ewm(alpha=1/ADX_PERIOD, min_periods=ADX_PERIOD).mean() / df['ATR'])
df['Minus_DI'] = 100 * (df['Minus_DM'].ewm(alpha=1/ADX_PERIOD, min_periods=ADX_PERIOD).mean() / df['ATR'])
df['DX'] = 100 * ((df['Plus_DI'] - df['Minus_DI']).abs() / (df['Plus_DI'] + df['Minus_DI']))
df['ADX'] = df['DX'].ewm(alpha=1/ADX_PERIOD, min_periods=ADX_PERIOD).mean()

# --- 5. Volumen-Analyse ---
VOL_WINDOW = 20
df['Vol_SMA'] = df['Volume'].rolling(window=VOL_WINDOW).mean()
df['Vol_Ratio'] = df['Volume'] / df['Vol_SMA']  # >1.5 = Spike

# ==========================================
# SCORING SYSTEM: 5 Indikatoren, je -1/0/+1
# ==========================================
df['Score_RSI'] = 0
df.loc[df['RSI'] < 35, 'Score_RSI'] = 1       # Überverkauft = bullish
df.loc[df['RSI'] > 65, 'Score_RSI'] = -1      # Überkauft = bearish

df['Score_MACD'] = 0
df.loc[df['MACD_Hist'] > 0, 'Score_MACD'] = 1     # MACD über Signal = bullish
df.loc[df['MACD_Hist'] < 0, 'Score_MACD'] = -1    # MACD unter Signal = bearish

df['Score_BB'] = 0
df.loc[df['BB_Pct'] < 0.15, 'Score_BB'] = 1       # Nahe unterem Band = bullish
df.loc[df['BB_Pct'] > 0.85, 'Score_BB'] = -1      # Nahe oberem Band = bearish

df['Score_ADX'] = 0
# Starker Aufwärtstrend: ADX > 20 und +DI > -DI
df.loc[(df['ADX'] > 20) & (df['Plus_DI'] > df['Minus_DI']), 'Score_ADX'] = 1
# Starker Abwärtstrend: ADX > 20 und -DI > +DI  
df.loc[(df['ADX'] > 20) & (df['Minus_DI'] > df['Plus_DI']), 'Score_ADX'] = -1

df['Score_Vol'] = 0
# Volumen-Spike bei steigendem Preis = bullish
df.loc[(df['Vol_Ratio'] > 1.3) & (df['Close'] > df['Close'].shift(1)), 'Score_Vol'] = 1
# Volumen-Spike bei fallendem Preis = bearish
df.loc[(df['Vol_Ratio'] > 1.3) & (df['Close'] < df['Close'].shift(1)), 'Score_Vol'] = -1

# Gesamtscore: Summe aller 5 Indikatoren (-5 bis +5)
df['Ensemble_Score'] = (df['Score_RSI'] + df['Score_MACD'] + df['Score_BB'] + 
                        df['Score_ADX'] + df['Score_Vol'])

# Signal: Kauf wenn Score >= 3 (mindestens 3 von 5 bullish)
#         Verkauf wenn Score <= -2 (mindestens 2 bearish)
KAUF_SCHWELLE = 3
VERKAUF_SCHWELLE = -2

df['Ensemble_Signal'] = 0
df.loc[df['Ensemble_Score'] >= KAUF_SCHWELLE, 'Ensemble_Signal'] = 1
df.loc[df['Ensemble_Score'] <= VERKAUF_SCHWELLE, 'Ensemble_Signal'] = -1

print(f"Multi-Signal Ensemble Konfiguration:")
print(f"  Indikatoren:      RSI, MACD, Bollinger, ADX+DI, Volumen")
print(f"  Kauf-Schwelle:    Score >= {KAUF_SCHWELLE} (von max. 5)")
print(f"  Verkauf-Schwelle: Score <= {VERKAUF_SCHWELLE} (von min. -5)")
print(f"  Hebel:            {LEVERAGE}x")
print(f"\nSignale berechnet ✓")
print(f"  Kaufsignale:      {(df['Ensemble_Signal'] == 1).sum()}")
print(f"  Verkaufssignale:  {(df['Ensemble_Signal'] == -1).sum()}")
print(f"\nScore-Verteilung:")
print(df['Ensemble_Score'].value_counts().sort_index())

Multi-Signal Ensemble Konfiguration:
  Indikatoren:      RSI, MACD, Bollinger, ADX+DI, Volumen
  Kauf-Schwelle:    Score >= 3 (von max. 5)
  Verkauf-Schwelle: Score <= -2 (von min. -5)
  Hebel:            2x

Signale berechnet ✓
  Kaufsignale:      0
  Verkaufssignale:  66

Score-Verteilung:
Ensemble_Score
-3      4
-2     62
-1    175
 0    249
 1    219
 2     43
Name: count, dtype: int64


## 8. Backtest-Engine: Alle Strategien mit $10.000 testen

In [13]:
def backtest(df, signal_col, start_capital=10_000, leverage=1, trailing_stop=None):
    """
    Simuliert Trading mit gegebenen Signalen.
    trailing_stop: z.B. 0.08 = Verkauf wenn Preis 8% unter dem höchsten Wert seit Kauf fällt
    """
    cash = start_capital
    shares = 0
    entry_price = 0
    highest_since_buy = 0
    portfolio_values = []
    buy_dates = []
    buy_prices = []
    sell_dates = []
    sell_prices = []
    trades = []
    
    for i, row in df.iterrows():
        price = float(row['Close'])
        signal = row[signal_col]
        
        # Trailing Stop-Loss prüfen
        if shares > 0 and trailing_stop and price > highest_since_buy:
            highest_since_buy = price
        
        stop_triggered = False
        if shares > 0 and trailing_stop:
            stop_level = highest_since_buy * (1 - trailing_stop)
            if price <= stop_level:
                stop_triggered = True
        
        # Kaufen: Alles investieren (mit Hebel)
        if signal == 1 and shares == 0 and cash > 0:
            shares = int((cash * leverage) / price)
            entry_price = price
            highest_since_buy = price
            cash = 0
            buy_dates.append(i)
            buy_prices.append(price)
            trades.append({'date': i, 'type': 'BUY', 'price': price, 'shares': shares, 'leverage': leverage})
        
        # Verkaufen: Signal oder Trailing Stop
        elif (signal == -1 or stop_triggered) and shares > 0:
            pnl_per_share = price - entry_price
            real_shares = shares / leverage
            total_pnl = pnl_per_share * shares
            cash = max(0, real_shares * entry_price + total_pnl)
            sell_dates.append(i)
            sell_prices.append(price)
            reason = 'STOP' if stop_triggered else 'SIGNAL'
            trades.append({'date': i, 'type': f'SELL ({reason})', 'price': price, 'shares': shares, 'pnl': total_pnl})
            shares = 0
            entry_price = 0
            highest_since_buy = 0
        
        # Portfolio-Wert
        if shares > 0:
            pnl = (price - entry_price) * shares
            real_shares = shares / leverage
            portfolio_value = max(0, real_shares * entry_price + pnl)
        else:
            portfolio_value = cash
        portfolio_values.append(portfolio_value)
    
    return {
        'values': pd.Series(portfolio_values, index=df.index),
        'buy_dates': buy_dates,
        'buy_prices': buy_prices,
        'sell_dates': sell_dates,
        'sell_prices': sell_prices,
        'trades': pd.DataFrame(trades) if trades else pd.DataFrame(),
        'final_value': portfolio_values[-1] if portfolio_values else start_capital
    }

# Alle Strategien backtesten
strategies = {
    'Linear Regression': ('LinReg_Signal', LEVERAGE, None),
    'Multi-Signal Ensemble': ('Ensemble_Signal', LEVERAGE, 0.08),  # mit 8% Trailing Stop
}

# NaN-Werte auf 0 setzen
for signal_col, _, _ in strategies.values():
    df[signal_col] = df[signal_col].fillna(0)

results = {}
for name, (signal_col, lev, ts) in strategies.items():
    results[name] = backtest(df, signal_col, STARTKAPITAL, leverage=lev, trailing_stop=ts)
    ret = (results[name]['final_value'] - STARTKAPITAL) / STARTKAPITAL * 100
    n_trades = len(results[name]['trades'])
    ts_str = f", TS={ts*100:.0f}%" if ts else ""
    print(f"{name:30s} ({lev}x{ts_str}) | Endwert: ${results[name]['final_value']:>10,.2f} | Return: {ret:>+7.2f}% | Trades: {n_trades}")

# Buy & Hold als Baseline — mit LEVERAGE Hebel
initial_price = float(df['Close'].iloc[0])
bh_shares = int((STARTKAPITAL * LEVERAGE) / initial_price)
bh_entry = initial_price
bh_real_shares = bh_shares / LEVERAGE
# Täglichen Portfolio-Wert mit Hebel berechnen
bh_values = pd.Series(
    [max(0, bh_real_shares * bh_entry + (p - bh_entry) * bh_shares) for p in df['Close']],
    index=df.index
)
bh_final = float(bh_values.iloc[-1])
bh_ret = (bh_final - STARTKAPITAL) / STARTKAPITAL * 100
print(f"{'Buy & Hold':30s} ({LEVERAGE}x)  | Endwert: ${bh_final:>10,.2f} | Return: {bh_ret:>+7.2f}% | Trades: 1")

# Trade-Details anzeigen
for name in results:
    if not results[name]['trades'].empty:
        print(f"\n--- {name} Trades ---")
        display(results[name]['trades'])

Linear Regression              (2x) | Endwert: $ 63,140.80 | Return: +531.41% | Trades: 13
Multi-Signal Ensemble          (2x, TS=8%) | Endwert: $ 10,000.00 | Return:   +0.00% | Trades: 0
Buy & Hold                     (2x)  | Endwert: $ 33,519.51 | Return: +235.20% | Trades: 1

--- Linear Regression Trades ---


,date,type,price,shares,leverage,pnl
0,2023-08-25,BUY,102.250000,195,2.0,NaN
1,2023-11-10,SELL (SIGNAL),118.589996,195,NaN,3186.299286
2,2024-04-04,BUY,165.830002,158,2.0,NaN
3,2024-07-08,SELL (SIGNAL),178.690002,158,NaN,2031.880096
4,2024-08-06,BUY,130.179993,232,2.0,NaN
5,2024-10-04,SELL (SIGNAL),170.899994,232,NaN,9447.040283
6,2024-12-10,BUY,127.739998,384,2.0,NaN
7,2025-03-24,SELL (SIGNAL),113.849998,384,NaN,-5333.759766
8,2025-04-08,BUY,78.209999,490,2.0,NaN
9,2025-05-12,SELL (SIGNAL),108.120003,490,NaN,14655.901794


## 9. Kauf/Verkauf-Signale auf Preischarts markieren

In [14]:
# --- Individuelle Charts pro Strategie mit Kauf/Verkauf-Signalen ---

def plot_strategy(name, result, df, indicator_traces=None, extra_subplot=None):
    """Plottet Preis + Kauf/Verkauf-Signale + optionale Indikatoren"""
    n_rows = 3 if extra_subplot else 2
    heights = [0.5, 0.25, 0.25] if extra_subplot else [0.7, 0.3]
    subtitles = [f'{SYMBOL} Preis mit {name} Signalen', 'Portfolio-Wert ($)']
    if extra_subplot:
        subtitles.append(extra_subplot['title'])
    
    fig = make_subplots(rows=n_rows, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        row_heights=heights, subplot_titles=subtitles)
    
    # Preischart
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Preis', 
                             line=dict(color='#2196F3', width=1.5)), row=1, col=1)
    
    if indicator_traces:
        for trace in indicator_traces:
            fig.add_trace(trace, row=1, col=1)
    
    # Kauf-Signale
    if result['buy_dates']:
        fig.add_trace(go.Scatter(
            x=result['buy_dates'], y=result['buy_prices'],
            mode='markers', name='Kauf',
            marker=dict(symbol='triangle-up', size=12, color='#4CAF50', 
                       line=dict(width=1, color='darkgreen'))
        ), row=1, col=1)
    
    # Verkauf-Signale
    if result['sell_dates']:
        fig.add_trace(go.Scatter(
            x=result['sell_dates'], y=result['sell_prices'],
            mode='markers', name='Verkauf',
            marker=dict(symbol='triangle-down', size=12, color='#F44336',
                       line=dict(width=1, color='darkred'))
        ), row=1, col=1)
    
    # Portfolio-Wert
    fig.add_trace(go.Scatter(x=df.index, y=result['values'], name='Portfolio',
                             line=dict(color='#FF9800', width=2)), row=2, col=1)
    fig.add_hline(y=STARTKAPITAL, line_dash="dash", line_color="gray", row=2, col=1,
                  annotation_text=f"Start: ${STARTKAPITAL:,}")
    
    # Extra Subplot (z.B. Score)
    if extra_subplot:
        for trace in extra_subplot['traces']:
            fig.add_trace(trace, row=3, col=1)
    
    final = result['final_value']
    ret = (final - STARTKAPITAL) / STARTKAPITAL * 100
    fig.update_layout(
        title=f'{name} — Endwert: ${final:,.2f} ({ret:+.2f}%) | {LEVERAGE}x Hebel',
        height=800 if extra_subplot else 600, template='plotly_dark',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode='x unified'
    )
    fig.update_yaxes(title_text="Preis ($)", row=1, col=1)
    fig.update_yaxes(title_text="Portfolio ($)", row=2, col=1)
    fig.show()

# 1. Linear Regression
plot_strategy('Linear Regression', results['Linear Regression'], df, [
    go.Scatter(x=df.index, y=df['Reg_Line'], name='Reg Line', line=dict(color='orange', width=1, dash='dash')),
    go.Scatter(x=df.index, y=df['Reg_Upper'], name='Upper Band', line=dict(color='red', width=0.8, dash='dot')),
    go.Scatter(x=df.index, y=df['Reg_Lower'], name='Lower Band', line=dict(color='green', width=0.8, dash='dot')),
])

# 2. Multi-Signal Ensemble (mit Score-Subplot)
plot_strategy('Multi-Signal Ensemble', results['Multi-Signal Ensemble'], df, 
    indicator_traces=[
        go.Scatter(x=df.index, y=df['BB_Upper'], name='BB Upper', line=dict(color='rgba(255,80,80,0.4)', width=0.8, dash='dot')),
        go.Scatter(x=df.index, y=df['BB_Lower'], name='BB Lower', line=dict(color='rgba(80,255,80,0.4)', width=0.8, dash='dot')),
    ],
    extra_subplot={
        'title': 'Ensemble Score (-5 bis +5)',
        'traces': [
            go.Bar(x=df.index, y=df['Ensemble_Score'], name='Score',
                   marker_color=np.where(df['Ensemble_Score'] >= 3, '#4CAF50', 
                                np.where(df['Ensemble_Score'] <= -2, '#F44336', '#666666'))),
            go.Scatter(x=df.index, y=[KAUF_SCHWELLE]*len(df), name=f'Kauf (≥{KAUF_SCHWELLE})',
                      line=dict(color='green', dash='dash', width=1)),
            go.Scatter(x=df.index, y=[VERKAUF_SCHWELLE]*len(df), name=f'Verkauf (≤{VERKAUF_SCHWELLE})',
                      line=dict(color='red', dash='dash', width=1)),
        ]
    }
)

# 3. Vergleich: Alle Strategien + Buy&Hold
fig_compare = go.Figure()
colors = {'Linear Regression': '#FF9800', 'Multi-Signal Ensemble': '#4CAF50'}
for name, result in results.items():
    fig_compare.add_trace(go.Scatter(x=df.index, y=result['values'], name=name,
                                      line=dict(color=colors.get(name, '#999'), width=2)))
fig_compare.add_trace(go.Scatter(x=df.index, y=bh_values, name='Buy & Hold',
                                  line=dict(color='#2196F3', width=1.5, dash='dash')))
fig_compare.add_hline(y=STARTKAPITAL, line_dash="dot", line_color="gray")
fig_compare.update_layout(title='Portfolio-Vergleich: Alle Strategien vs Buy & Hold',
                           height=500, template='plotly_dark', hovermode='x unified',
                           yaxis_title='Portfolio-Wert ($)')
fig_compare.show()